In [ ]:
# pip install tensorflow==2.5
# pip install keras==2.6
# pip install qiskit
# pip install yfinance
# pip install matplotlib
# pip install pennylane
# pip install sklearn
# pip install qiskit-machine-learning

In [2]:
# import tensorflow as tf
# import qiskit as qk
# import yfinance as yf
# import numpy as np
# import matplotlib.pyplot as plt
# from qiskit import QuantumCircuit
# from qiskit_aer import Aer
# from qiskit.visualization import plot_histogram
# from keras.models import Sequential
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense, Input
# from tensorflow.keras.optimizers import Adam
# from tensorflow.keras.optimizers import RMSprop
# from tensorflow.keras import backend as K
# import pennylane as qml
# from pennylane import numpy as np

# from sklearn.preprocessing import MinMaxScaler
# from qiskit.opflow import AerPauliExpectation, CircuitSampler, StateFn
# from qiskit_machine_learning.algorithms.distribution_learners.qgan import QGAN

import tensorflow as tf
import qiskit as qk
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram
from keras.models import Sequential
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras import backend as K
import pennylane as qml
from pennylane import numpy as np

from sklearn.preprocessing import MinMaxScaler
from qiskit.primitive import AerPauliExpectation
from qiskit.quantum_info import AerPauliExpectation, CircuitSampler, StateFn
from qiskit_machine_learning.algorithms.distribution_learners.qgan import QGAN

ModuleNotFoundError: No module named 'qiskit.primitive'

In [3]:
ticker = "AAPL"
start_date = "2015-01-01"
end_date = "2023-09-16"
stock_data = yf.download(ticker, start=start_date, end = end_date)

[*********************100%***********************]  1 of 1 completed


In [4]:
q = qk.QuantumRegister(1)
c = qk.ClassicalRegister(1)
qc = QuantumCircuit(q,c)
qc.h(q[0])
qc.measure(q[0], c[0])

In [ ]:
def build_generator():
    model = Sequential()
    model.add(Dense(128, input_dim=1, activation='relu'))
    model.add(Dense(256, activation = 'relu'))
    model.add(Dense(512, activation = 'relu'))
    model.add(Dense(1))
    model.summary()
    noise = Input(shape=(1,))
    gen_data = model(noise)
    return tf.keras.models.Model(noise, gen_data)


def build_discriminator():
    model = Sequential()
    model.add(Dense(512, input_dim=1, activation='relu'))
    model.add(Dense(256, activation='relu'))
    model.add(Dense(128, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.summary()
    real_data = Input(shape=(1,))
    validity = model(real_data)
    return tf.keras.models.Model(real_data, validity)

# Define Qgan model
def build_qgan(generator, discriminator):
    discriminator.trainable = False
    model = Sequential()
    model.add(generator)
    model.add(discriminator)
    optimizer = Adam(learning_rate=0.0002)
    model.compile(loss='binary_crossentropy', optimizer=optimizer)
    return model

generator = build_generator()
discriminator = build_discriminator()
qgan = build_qgan(generator, discriminator)

discriminator.compile(optimizer=Adam(learning_rate=0.0002), loss ='binary_crossentropy')

#Train QGAN
num_epochs = 2000
batch_size = 64

for epoch in range(num_epochs):
    noise = np.random.normal(0, 1, (batch_size, 1))
    gen_data = generator.predict(noise)
    real_data = stock_data['Adj Close'].values.reshape((-1, 1))[np.random.randint(0, len(stock_data), batch_size)]
    X = np.concatenate((real_data, gen_data))
    y = np.zeros((2*batch_size, 1))
    y[:batch_size, :] = 1
    discriminator.trainable = True
    d_loss = discriminator.train_on_batch(X, y)
    noise = np.random.normal(0, 1, (batch_size, 1))
    y = np.ones((batch_size, 1))
    discriminator.trainable = False
    g_loss = qgan.train_on_batch(noise, y)
    if epoch % 1000 == 0:
        print("Epoch", epoch, "D loss:", d_loss, "G loss:", g_loss)
        
# Generate synthetic data
num_samples = len(stock_data)
noise = np.random.normal(0, 1, (num_samples, 1))
gen_data = generator.predict(noise)
gen_data = np.squeeze(gen_data)

# Plot real and synthetic data
plt.figure(figsize=(16, 6))
plt.plot(stock_data.index, stock_data["Adj Close"], label="Real Data")
plt.plot(stock_data.index, gen_data, label="Synthetic Data")
plt.legend()
plt.title(f"{ticker} Stock Price")
plt.xlabel("Date")
plt.ylabel("Price ($)")
plt.show()

/Users/kartikkaramore/anaconda3/envs/tensorflow-env/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_16 (Dense)                │ (None, 128)            │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 165,377 (646.00 KB)

 Trainable params: 165,377 (646.00 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_20 (Dense)                │ (None, 512)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 165,377 (646.00 KB)

 Trainable params: 165,377 (646.00 KB)

 Non-trainable params: 0 (0.00 B)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Epoch 0 D loss: 0.43598992 G loss: [array(0.43598992, dtype=float32), array(0.43598992, dtype=float32)]
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
2/2 ━━━

In [2]:
# Define QGAN model with Wasserstein loss
def build_qgan(generator, discriminator):
    discriminator.trainable = False
    model = Sequential()
    model.add(generator)
    model.add(discriminator)
    optimizer = RMSprop(learning_rate=0.00005)
    model.compile(loss=wasserstein_loss, optimizer=optimizer)
    return model

def wasserstein_loss(y_true, y_pred):
    return K.mean(y_true * y_pred)

generator = build_generator()
discriminator = build_discriminator()
qgan = build_qgan(generator, discriminator)

discriminator.compile(optimizer=RMSprop(learning_rate=0.00005), loss=wasserstein_loss)

# Train QGAN
num_epochs = 2000
batch_size = 64

for epoch in range(num_epochs):
    noise = np.random.normal(0, 1, (batch_size, 1))
    gen_data = generator.predict(noise)
    real_data = stock_data["Adj Close"].values.reshape((-1, 1))[np.random.randint(0, len(stock_data), batch_size)]
    X = np.concatenate((real_data, gen_data))
    y = np.ones((2*batch_size, 1))
    y[:batch_size, :] = -1
    discriminator.trainable = True
    d_loss = discriminator.train_on_batch(X, y)
    noise = np.random.normal(0, 1, (batch_size, 1))
    y = np.ones((batch_size, 1))
    discriminator.trainable = False
    g_loss = qgan.train_on_batch(noise, y)
    if epoch % 1000 == 0:
        print("Epoch", epoch, "D loss:", d_loss, "G loss:", g_loss)

# Generate synthetic data
num_samples = len(stock_data)
noise = np.random.normal(0, 1, (num_samples, 1))
gen_data = generator.predict(noise)
gen_data = np.squeeze(gen_data)

# Plot real and synthetic data
plt.figure(figsize=(16, 6))
plt.plot(stock_data.index, stock_data["Adj Close"], label="Real Data")
plt.plot(stock_data.index, gen_data, label="Synthetic Data")
plt.legend()
plt.title(f"{ticker} Stock Price")
plt.xlabel("Date")
plt.ylabel("Price ($)")
plt.show()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 165,377 (646.00 KB)

 Trainable params: 165,377 (646.00 KB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 512)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 165,377 (646.00 KB)

 Trainable params: 165,377 (646.00 KB)

 Non-trainable params: 0 (0.00 B)

NameError: name 'RMSprop' is not defined

In [ ]:
def build_generator():
    model = Sequential()
    model.add(Dense(128, input_dim=1, activation='relu'))
    model.add(Dense(256, activation='relu'))
    model.add(Dense(512, activation='relu'))
    model.add(Dense(1))
    model.summary()
    noise = Input(shape=(1,))
    gen_data = model(noise)
    return tf.keras.models.Model(noise, gen_data)

def build_discriminator():
    model = Sequential()
    model.add(Dense(512, input_dim=1, activation='relu'))
    model.add(Dense(256, activation='relu'))
    model.add(Dense(128, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    model.summary()
    real_data = Input(shape=(1,))
    validity = model(real_data)
    return tf.keras.models.Model(real_data, validity)

def build_qgan(generator, discriminator):
    discriminator.trainable = False
    model = Sequential()
    model.add(generator)
    model.add(discriminator)
    optimizer = Adam(0.0002, 0.5)
    model.compile(loss='binary_crossentropy', optimizer=optimizer)
    return model

generator = build_generator()
discriminator = build_discriminator()
qgan = build_qgan(generator, discriminator)

discriminator.compile(optimizer='adam', loss='binary_crossentropy')

num_epochs = 20000
batch_size = 32

for epoch in range(num_epochs):
    noise = np.random.normal(0, 1, (batch_size, 1))
    gen_data = generator.predict(noise)
    real_data = stock_data["Adj Close"].values.reshape((-1, 1))[np.random.randint(0, len(stock_data), batch_size)]
    X = np.concatenate((real_data, gen_data))
    y = np.zeros((2*batch_size, 1))
    y[:batch_size, :] = 1
    discriminator.trainable = True
    d_loss = discriminator.train_on_batch(X, y)
    noise = np.random.normal(0, 1, (batch_size, 1))
    y = np.ones((batch_size, 1))
    discriminator.trainable = False
    g_loss = qgan.train_on_batch(noise, y)
    if epoch % 1000 == 0:
        print("Epoch", epoch, "D loss:", d_loss, "G loss:", g_loss)
        
        num_samples = len(stock_data)
        noise = np.random.normal(0, 1, (num_samples, 1))
        gen_data = generator.predict(noise)
        gen_data = np.squeeze(gen_data)

        plt.figure(figsize=(16, 6))
        plt.plot(stock_data.index, stock_data["Adj Close"], label="Real Data")
        plt.plot(stock_data.index, gen_data, label="Synthetic Data")
        plt.legend()
        plt.title(f"{ticker} Stock Price")
        plt.xlabel("Date")
        plt.ylabel("Price ($)")
        plt.show()

num_samples = len(stock_data)
noise = np.random.normal(0, 1, (num_samples, 1))
gen_data = generator.predict(noise)
gen_data = np.squeeze(gen_data)

plt.figure(figsize=(16, 6))
plt.plot(stock_data.index, stock_data["Adj Close"], label="Real Data")
plt.plot(stock_data.index, gen_data, label="Synthetic Data")
plt.legend()
plt.title(f"{ticker} Stock Price")
plt.xlabel("Date")
plt.ylabel("Price ($)")
plt.show()